# Semantic Tool Router — End-to-End Demo

Pipeline: **registry → retrieve top-k → agent selects tool → token savings**

```bash
pip install semantic-tool-router[sentence-transformers]
```

In [ ]:
from semantic_tool_router import ToolRegistry, ToolRouter
from semantic_tool_router.agent_eval import RankOneSelector, evaluate_agent
from semantic_tool_router.evaluation import BenchmarkTask
from semantic_tool_router.mcp import estimate_tokens

registry = ToolRegistry.from_file("examples/tools.json")
router = ToolRouter(registry)  # fast profile: hashing + BM25
query = "debug failing github ci logs"
top_k = 3

results = router.discover(query, top_k=top_k)
print(f"Query: {query}\n")
for i, item in enumerate(results, start=1):
    print(f"{i}. {item.tool.name} ({item.score:.3f})")
    print(f"   {item.tool.description}")

In [ ]:
# Downstream agent: pick the top-ranked tool
task = BenchmarkTask(
    query=query,
    expected_tools=("github_fetch_workflow_logs",),
)
report = evaluate_agent(
    router,
    [task],
    top_k=top_k,
    selector=RankOneSelector(),
    selector_name="rank1",
)
print(report.as_dict())

In [ ]:
# Context savings: all tools vs retrieved subset
all_tools = [
    {
        "name": tool.name,
        "description": tool.description,
        "inputSchema": tool.input_schema,
    }
    for tool in registry.tools()
]
selected = [
    {
        "name": item.tool.name,
        "description": item.tool.description,
        "inputSchema": item.tool.input_schema,
    }
    for item in results
]
all_tokens = estimate_tokens(all_tools)
selected_tokens = estimate_tokens(selected)
saved = 1.0 - (selected_tokens / all_tokens)
print(f"All tools:       {all_tokens} tokens")
print(f"Selected top-{top_k}: {selected_tokens} tokens")
print(f"Context saved:   {saved:.1%}")

## Production-quality routing

Use the CLI quality profile (MiniLM + cross-encoder, no BM25 dilution):

```bash
python -m semantic_tool_router discover "read the project README" \
  --registry examples/tools.json \
  --profile quality
```